## 2024-10-11: Feature Selection

### Authors
* Nicole Tin (nicole@velexi.com)


### Overview
This Jupyter notebook is intended to explore

* various packages/architectures for constructing a CNN to use on images

### Key Results

The key results of this experiment are ...
* choosing by high pearson R to the y variable and by top shap values yield similar results, but do not improve on the full feature set.

### Future Ideas
* benchmark with professional dermatologists estimate (+-10 yrs)
  * professional skin attribute assessment
* inter-group variability
* ML nor doctors may not be good at identifying biological age
* scientifically, there may be more group variability
* one person in training/test split as L/R hands
  * should it be strictly L or R?
  * train only on L, test on R, would outcomes be perfect? high error? 


### EDA ideas (age vs sun exposure)
* L vs. R
* inter age

### Experiment Parameters

In [52]:
# --- Experiment parameters

# Name of experiment. Used for MLflow experiment name, output files, etc.
experiment_name = "example-experiment"

# Paths
# src_dir = '/Users/nicole/Documents/DermaML_local/hawkeye-hands-2024-07-29'
# image_dir = '/processed images/'
# csv_file = '/metadata.csv'
DATA = '2024-08-01_NT_Hawkeye-Hands-Texture-Features.csv'

# ------ Algorithm parameters

# train/test split
split = 0.7
epochs = 10
learning_rate = 1e3
random_seed = 42

num_best_models = 5

### Preparations

In [41]:
# --- Imports

# Internal library
# import bin.calculate_hand as loading

# Standard library
# from random import random
# import math
# import shutil
# import typing
# from datetime import datetime
# from pathlib import Path

# External packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

# from tensorflow import keras
# from tensorflow.python.keras import layers

# Entropy
# random.random_seed(42)
# np.random.random_seed(42)

np.set_printoptions(precision=3, suppress=True)

### Preparations 2

In [11]:
import pandas as pd

df = pd.read_csv(DATA).drop(columns=['Unnamed: 0'])
y = df.iloc[:, -1]
X = df.iloc[:, :-1]

### High Pearson R

In [31]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

# define feature selection
fs = SelectKBest(score_func=f_regression, k=10)
# apply feature selection
X_selected = fs.fit_transform(X, y)
pearson_columns = fs.get_support(True)
print(X_selected.shape)

(576, 10)


In [64]:
f_statistic, p_values = f_regression(X, y)

# pearsons correlation as f-score
pearson = pd.DataFrame(data=[X.columns, f_statistic, p_values,],)
pearson = pearson.T
pearson.columns = columns = ['feature', 'f-stat', 'p-value']
pearson.sort_values('f-stat')

,feature,f-stat,p-value
6,GLCM_InverseDifferenceMoment_Mean_wrinkles_pyf...,0.00483,0.944616
2,GLCM_ASM_Mean_wrinkles_pyfeats,0.065294,0.798408
29,LTE_EE_7_lte,0.300342,0.583881
26,NGTDM_Complexity_ngtdm,0.56619,0.452085
13,GLCM_Information1_Mean_wrinkles_pyfeats,0.83625,0.360855
33,LTE_LS_7_lte,0.861853,0.353611
0,relative_redness_mean,1.230938,0.267689
32,LTE_ES_7_lte,1.750458,0.186346
12,GLCM_DifferenceEntropy_Mean_wrinkles_pyfeats,1.787573,0.181752
11,GLCM_DifferenceVariance_Mean_wrinkles_pyfeats,1.918252,0.166588


In [65]:
pearson_columns

array([ 5,  8, 15, 24, 28, 37, 38, 39, 40, 45])

In [72]:
from pycaret import regression

# --- Perform AutoML Evaluation

# Set up the dataset for AutoML regression
regression.setup(data=df.iloc[:, np.append(pearson_columns, -1)],
                 target="Age",
                 log_experiment=True,
                 experiment_name=experiment_name,
                 session_id=random_seed,
                ) 

best_models = regression.compare_models(n_select=num_best_models, verbose=False)

,Description,Value
0,Session id,42
1,Target,Age
2,Target type,Regression
3,Original data shape,"(576, 11)"
4,Transformed data shape,"(576, 11)"
5,Transformed train set shape,"(403, 11)"
6,Transformed test set shape,"(173, 11)"
7,Numeric features,10
8,Preprocess,True
9,Imputation type,simple


/Users/nicole/Documents/GitHub/DermaML/.direnv/python-3.11/lib/python3.11/site-packages/pycaret/internal/pycaret_experiment/supervised_experiment.py:339: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_cols, subset=["TT (Sec)"])
2024/10/17 22:03:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/17 22:03:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/17 22:03:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/17 22:03:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` param

In [73]:
regression.pull()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,13.1856,1522.4214,24.9368,0.4968,0.3769,0.3521,0.091
ada,AdaBoost Regressor,15.5906,1569.8438,26.5012,0.3919,0.4441,0.5015,0.039
lightgbm,Light Gradient Boosting Machine,14.8386,1629.7156,27.8785,0.2862,0.4051,0.4018,0.211
lr,Linear Regression,16.8389,1654.1089,28.7157,0.2032,0.4545,0.4531,0.024
rf,Random Forest Regressor,13.9457,1680.5946,28.2137,0.2022,0.3932,0.3893,0.176
et,Extra Trees Regressor,13.4661,1634.9754,27.6753,0.1971,0.3673,0.3389,0.111
dt,Decision Tree Regressor,16.4550,1708.0079,29.8614,0.1403,0.5161,0.4292,0.015
ridge,Ridge Regression,18.8820,1754.0692,30.2335,0.1230,0.4896,0.5381,0.012
llar,Lasso Least Angle Regression,19.2077,1767.4381,30.5052,0.1009,0.4964,0.5481,0.013
lasso,Lasso Regression,19.2742,1764.5879,30.5134,0.0990,0.4966,0.5470,0.013


### VIF

In [74]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

test_df = df.iloc[:, np.append(pearson_columns, -1)]
vif = pd.DataFrame(data=[test_df.columns]).T
vif['vif'] = [variance_inflation_factor(test_df.values, i) for i in range(len(test_df.columns))]
vif.sort_values('vif')

,0,vif
10,Age,2.376349
9,contrast_scikit,10.565803
3,NGTDM_Contrast_ngtdm,32.480762
2,GLCM_MaximalCorrelationCoefficient_Mean_wrinkl...,87.248353
4,LTE_LL_7_lte,195.239261
8,lbp_6,1723.918135
5,lbp_3,1780.770811
7,lbp_5,2686.169271
6,lbp_4,2728.764083
1,GLCM_SumVariance_Mean_wrinkles_pyfeats,164841.083719


### Top SHAP 

In [80]:
top_shap = ["lbp_5", "lbp_4", "lbp_6", "relative_redness_std", "lbp_3", "GLCM_SumEntropy_Mean_wrinkles_pyfeats", "skin_folds_hessian", "correlation_scikit", "Age"]

# --- Perform AutoML Evaluation

# Set up the dataset for AutoML regression
regression.setup(data=df[top_shap],
                 target="Age",
                 log_experiment=True,
                 experiment_name=experiment_name,
                 session_id=random_seed,
                ) 

best_models = regression.compare_models(n_select=num_best_models, verbose=False)

,Description,Value
0,Session id,42
1,Target,Age
2,Target type,Regression
3,Original data shape,"(576, 9)"
4,Transformed data shape,"(576, 9)"
5,Transformed train set shape,"(403, 9)"
6,Transformed test set shape,"(173, 9)"
7,Numeric features,8
8,Preprocess,True
9,Imputation type,simple


/Users/nicole/Documents/GitHub/DermaML/.direnv/python-3.11/lib/python3.11/site-packages/pycaret/internal/pycaret_experiment/supervised_experiment.py:339: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_cols, subset=["TT (Sec)"])
2024/10/17 22:16:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/17 22:16:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/17 22:16:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/10/17 22:16:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` param

In [81]:
regression.pull()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lr,Linear Regression,13.7907,1516.0615,25.0771,0.4851,0.3858,0.3785,0.981
lightgbm,Light Gradient Boosting Machine,13.0666,1570.2321,25.9887,0.4220,0.3910,0.3884,0.462
ridge,Ridge Regression,15.8545,1625.3763,27.4278,0.3349,0.4463,0.5023,0.019
lar,Least Angle Regression,15.5120,1617.8470,28.1339,0.2678,0.4635,0.4168,0.012
lasso,Lasso Regression,16.8014,1678.8506,28.6764,0.2471,0.4843,0.5477,0.016
llar,Lasso Least Angle Regression,16.8014,1678.8497,28.6763,0.2471,0.4843,0.5477,0.022
en,Elastic Net,17.8682,1726.7018,29.7135,0.1675,0.4951,0.5702,0.021
br,Bayesian Ridge,17.8389,1682.3652,29.7470,0.1495,0.4956,0.5677,0.013
omp,Orthogonal Matching Pursuit,20.5191,1834.7219,31.3872,0.0411,0.5619,0.6415,0.031
et,Extra Trees Regressor,13.0772,1722.3744,28.6949,-0.0231,0.3495,0.3171,0.100
